# FLARE

- Region-Aware Deep Learning Framework for Urban Flood Risk Estimation Using Surveillance Cameras (2026)

<p align="center">
<img src="./utils/images/main_figure.png" style="max-width:30%; height:auto;" alt="error"></img>
</p>

In [ ]:
import numpy as np
import cv2
import subprocess
import os
import sys
import json
import matplotlib.pyplot as plt
from statistics import mean

from scipy.stats import hmean
from PIL import Image
from PIL import ImageFont, ImageDraw, Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import VGG

from hydra.core.global_hydra import GlobalHydra
from hydra import initialize, compose
# from utils.sam2.build_sam import build_sam2
from utils.sam2.sam2.build_sam import build_sam2
# from utils.sam2.sam2_image_predictor import SAM2ImagePredictor
from utils.sam2.sam2.sam2_image_predictor import SAM2ImagePredictor

## Data Acquisition

- From 'input_video' folder

In [ ]:
extracted_frame = []
target_sec = [0, 1800] # any number in seconds

## With video, start from here

# video_file = "./input_video/name-of-your-video.avi"

# def save_frame_at_second(video_path, output_path, target_sec):
#     cap = cv2.VideoCapture(video_path)
#     if not cap.isOpened():
#         print("No video available.")
#         return
#     fps = cap.get(cv2.CAP_PROP_FPS)
#     target_frame = int(target_sec * fps)
#     cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
#     ret, frame = cap.read()
#     if ret:
#         cv2.imwrite(output_path, frame)
#         print(f"An frame image of {target_sec}s is saved at {output_path}.")
#     else:
#         print("No frames available.")
#     cap.release()

# for i in range(2):
#     output_file = f"./input_video/image{i+1}.png"
#     save_frame_at_second(video_file, output_file, target_sec[i])

In [ ]:
# With pictures, start from here

PATH1 = './input_video/image1.png'
PATH2 = './input_video/image2.png'
extracted_frame.append(PATH1)
extracted_frame.append(PATH2)
images = []     
images_bgr = []

for path in extracted_frame:
    img = cv2.imread(path)
    if img is None:
        print(f"Failed to load {path}")
    else:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGBA)
        images.append(img_rgb) 
        images_bgr.append(img)

if images:
    fig, axes = plt.subplots(1, len(images), figsize=(15 * len(images), 15))
    
    if len(images) == 1:
        axes = [axes]
    for idx, img in enumerate(images):
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f'Image {idx + 1}')
    
    plt.show()
else:
    print("No images to show.")


img_1 = images_bgr[0].copy()
img_n = images_bgr[1].copy()
img_1_org = img_1.copy()
img_n_org = img_n.copy()
img_height, img_width = img_n_org.shape[:2]

std_cor = []
seg_cor_1 = []
seg_cor_2 = []
ROI_cor = []

In [ ]:
def click_event_seg(event, x, y, flags, param):
    global seg_cor_1
    if event == cv2.EVENT_LBUTTONDOWN:
        cv2.putText(img_1, f'({x},{y})', (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0),  2, cv2.LINE_AA)
        cv2.circle(img_1, (x, y), 8, (255, 0, 0), -1)  
        cv2.imshow('image', img_1)
        seg_cor_1.append(x)
        seg_cor_1.append(y)
    
    elif event == cv2.EVENT_RBUTTONDOWN:
        org = (int((img_width - 200) / 5), int(img_height - 200))
        cv2.putText(img_1, "no segment point 1", org, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        cv2.imshow('image', img_1)
        seg_cor_1 = None

def click_event_seg_2(event, x, y, flags, param):
    global seg_cor_2
    if event == cv2.EVENT_LBUTTONDOWN:
        cv2.putText(img_n, f'({x},{y})', (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0),  2, cv2.LINE_AA)
        cv2.circle(img_n, (x, y), 8, (255, 0, 0), -1)  
        cv2.imshow('image', img_n)
        seg_cor_2.append(x)
        seg_cor_2.append(y)

    elif event == cv2.EVENT_RBUTTONDOWN:
        org = (int((img_width - 200) / 5), int(img_height - 300))
        cv2.putText(img_1, "no segment point 2", org, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        cv2.imshow('image', img_1)
        seg_cor_2 = None

def click_event_std(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        cv2.putText(img_1, f'({x},{y})', (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (60, 200, 235),  2, cv2.LINE_AA)
        cv2.circle(img_1, (x, y), 8, (60, 200, 235), -1)
        cv2.imshow('image', img_1)
        std_cor.append(x)
        std_cor.append(y)

drawing = False

def draw_rectangle(event, x, y, flags, param):
    global drawing, water_start_x, water_start_y, water_width, water_height, water_final_x, water_final_y, img_1_copy

    if event == cv2.EVENT_RBUTTONDOWN:
        img_1_copy = img_1.copy()
        org = (int((img_width - 200) / 5), int(img_height - 400))
        cv2.putText(img_1_copy, "no puddle region 1", org, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        cv2.imshow('image', img_1_copy)
        water_start_x = None
        water_start_y = None

    elif event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        water_start_x = x
        water_start_y = y

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            img_1_copy = img_1.copy()
            cv2.rectangle(img_1_copy, (water_start_x, water_start_y), (x, y), (0, 255, 0), 5)
            cv2.imshow('image', img_1_copy)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        water_final_x = min(water_start_x, x)
        water_final_y = min(water_start_y, y)
        water_width = abs(water_start_x - x)
        water_height = abs(water_start_y - y)

        top_left = (water_final_x, water_final_y)
        top_right = (water_final_x + water_width, water_final_y)
        bottom_left = (water_final_x, water_final_y + water_height)
        bottom_right = (water_final_x + water_width, water_final_y + water_height)

        ROI_cor.append(top_left)
        ROI_cor.append(top_right)
        ROI_cor.append(bottom_left)
        ROI_cor.append(bottom_right)

def draw_rectangle2(event, x, y, flags, param):
    global drawing, water_start_x, water_start_y, water_width, water_height, water_final_x, water_final_y, img_1_copy, img_1_copy2
    if event == cv2.EVENT_RBUTTONDOWN:
        img_1_copy2 = img_1_copy.copy()
        org = (int((img_width - 200) / 5), int(img_height - 500))
        cv2.putText(img_1_copy2, "no puddle region 2", org, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        cv2.imshow('image', img_1_copy2)
        water_start_x = None
        water_start_y = None

    elif event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        water_start_x = x
        water_start_y = y

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            img_1_copy2 = img_1_copy.copy()
            cv2.rectangle(img_1_copy2, (water_start_x, water_start_y), (x, y), (0, 255, 0), 5)
            cv2.imshow('image', img_1_copy2)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        water_final_x = min(water_start_x, x)
        water_final_y = min(water_start_y, y)
        water_width = abs(water_start_x - x)
        water_height = abs(water_start_y - y)

        top_left = (water_final_x, water_final_y)
        top_right = (water_final_x + water_width, water_final_y)
        bottom_left = (water_final_x, water_final_y + water_height)
        bottom_right = (water_final_x + water_width, water_final_y + water_height)

        ROI_cor.append(top_left)
        ROI_cor.append(top_right)
        ROI_cor.append(bottom_left)
        ROI_cor.append(bottom_right)
            
def convert_and_put_text(org_image, text, num):
    image_convert = org_image
    image_pil = Image.fromarray(image_convert)
    draw = ImageDraw.Draw(image_pil)
    draw.text((20, (img_height - 140)),  text , font=ImageFont.truetype("./utils/font/NanumSquareB.ttf", 50), fill=(255,255,255))
    image_convert_2_numpy = np.array(image_pil)
    return image_convert_2_numpy

In [ ]:
cv2.imshow('image', convert_and_put_text(img_1, "1. Please click one Anchor Point, then press any key.", 1))
cv2.setMouseCallback('image', click_event_std)
cv2.waitKey(0)

cv2.imshow('image', convert_and_put_text(img_1, "2. Please click one reference point for water segmentation, then press any key.", 2))
cv2.setMouseCallback('image', click_event_seg)
cv2.waitKey(0)

cv2.imshow('image', convert_and_put_text(img_n, "3. Please click one reference point for water segmentation, then press any key.", 3))
cv2.setMouseCallback('image', click_event_seg_2)
cv2.waitKey(0)

cv2.imshow('image', convert_and_put_text(img_1, "4. Please drag over the first puddle, then press any key.", 4))
cv2.setMouseCallback('image', draw_rectangle)
cv2.waitKey(0)

cv2.imshow('image', convert_and_put_text(img_1_copy, "5. Please drag over the second puddle, then press any key.", 5))
cv2.setMouseCallback('image', draw_rectangle2)
cv2.waitKey(0)

cv2.destroyAllWindows()

## Preprocessing

### Object Detection

In [ ]:
cropped_car_images_1 = []
cropped_car_images_2 = []
for path in [PATH1,PATH2]:
    command = [
        sys.executable,
        './utils/YOLOv6/infer.py',
        '--weights', './utils/model_weights/yolov6l6.pt',
        '--source', path,
        '--yaml', './utils/YOLOv6/data/coco.yaml',
        '--device', '0',
        '--save-txt', 
        '--save-dir', './outputs/yolo/car'
    ]

    subprocess.run(command)

 
    file_path = f"./outputs/yolo/car/labels/{path.split('/')[-1].split('.')[0]}.txt"
    result_image_path = f"./outputs/yolo/car/{path.split('/')[-1].split('.')[0]}.png"
    result = []


    if os.path.exists(file_path):
        with open(file_path, 'r') as file:
            for line in file:
                elements = line.strip().split()
                int_elements = list(map(float, elements))
                result.append(int_elements)
    else:
        print("No cars detected.")
        result = []
        car_detected_img = None

    img_height, img_width = img_1_org.shape[:2]
    if len(result) != 0:
        
        unique_boxes = set()
        for line in result:
            class_id = line[0]
            confidence = line[5]
            if class_id == 2 and confidence >= 0.4:
                x_center, y_center, width, height = line[1:5]

                x_center *= img_width
                y_center *= img_height
                width *= img_width
                height *= img_height

                x_min = int(x_center - width / 2)
                y_min = int(y_center - height / 2)
                x_max = int(x_center + width / 2)
                y_max = int(y_center + height / 2)

                corners = ((x_min, y_min), (x_max, y_min), (x_max, y_max), (x_min, y_max))
                if corners not in unique_boxes:
                    unique_boxes.add(corners)
                    if path.split('/')[-1].split('.')[0][-1] == '1':
                        cropped_img = img_1_org[y_min:y_max, x_min:x_max]
                        cropped_car_images_1.append(cropped_img)
                        car_detected_img1 = cv2.imread(f"./outputs/yolo/car/{path.split('/')[-1].split('.')[0]}.png", cv2.IMREAD_COLOR)
                        detected_img1 = cv2.cvtColor(car_detected_img1, cv2.COLOR_BGR2RGB)
                        plt.imshow(detected_img1)
                        plt.show()
                    else:
                        cropped_img = img_n_org[y_min:y_max, x_min:x_max]
                        cropped_car_images_2.append(cropped_img)
                        car_detected_img2 = cv2.imread(f"./outputs/yolo/car/{path.split('/')[-1].split('.')[0]}.png", cv2.IMREAD_COLOR)
                        detected_img2 = cv2.cvtColor(car_detected_img2, cv2.COLOR_BGR2RGB)
                        plt.imshow(detected_img2)
                        plt.show()

### Water Segmentation

In [ ]:
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Model moved to: {device}")

config_path = "./utils/sam2/sam2/configs/sam2.1/"
config_name = "sam2.1_hiera_l.yaml"
sam_checkpoint = "./utils/model_weights/sam2.1_hiera_large.pt"

with initialize(config_path=config_path, job_name="sam2", version_base="2.1"):
    cfg = compose(config_name=config_name)
    sam_model = build_sam2(config_file=config_name, ckpt_path=sam_checkpoint, device=device)
sam_model.to(device)
predictor = SAM2ImagePredictor(sam_model)

input1 = img_1_org
input2 = img_n_org
image1 = cv2.cvtColor(input1, cv2.COLOR_BGR2RGB)
image2 = cv2.cvtColor(input2, cv2.COLOR_BGR2RGB)
input_point = np.array([seg_cor_1])
input_point_2 = np.array([seg_cor_2])
input_label_1 = np.array([1])
input_label_2 = np.array([1])

if len(seg_cor_1) == 0:
    img_1_mask = np.zeros((img_height, img_width), dtype=np.uint8)
else:
    predictor.set_image(image1)
    masks, scores, logits = predictor.predict(
        point_coords=input_point,
        point_labels=input_label_1,
        multimask_output=True,
    )

    img_1_mask = (masks[1] * 255).astype(np.uint8)

if len(seg_cor_2) == 0:
    img_n_mask = np.zeros((img_height, img_width), dtype=np.uint8)
else:
    predictor.set_image(image2)
    masks, scores, logits = predictor.predict(
        point_coords=input_point_2,
        point_labels=input_label_2,
        multimask_output=True,
    )
    img_n_mask = (masks[1] * 255).astype(np.uint8)

img_1_org_show = cv2.cvtColor(img_1_org, cv2.COLOR_BGR2RGB)
img_n_org_show = cv2.cvtColor(img_n_org, cv2.COLOR_BGR2RGB)
img_1_copy2_show = cv2.cvtColor(img_1_copy2, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(1, 5, figsize=(25, 5))

axes[0].imshow(img_1_org_show)
axes[0].axis('off')
axes[0].set_title('first frame original image')

axes[1].imshow(img_1_copy2_show)
axes[1].axis('off')
axes[1].set_title('ROI plotted')

axes[2].imshow(img_1_mask, cmap = 'gray')
axes[2].axis('off')
axes[2].set_title('first frame water mask')

axes[3].imshow(img_n_org_show)
axes[3].axis('off')
axes[3].set_title('second frame original image')

axes[4].imshow(img_n_mask, cmap = 'gray')
axes[4].axis('off')
axes[4].set_title('second frame water mask')

plt.tight_layout()
plt.show()

In [ ]:
def mask_add(image, mask, color):
    
    purple_mask = np.zeros_like(image)
    purple_mask[mask > 0] = color

    alpha_channel = np.zeros_like(mask)
    alpha_channel[mask > 0] = 255
    purple_mask = cv2.merge((*cv2.split(purple_mask), alpha_channel))

    original_image = cv2.cvtColor(image, cv2.COLOR_BGR2BGRA)

    composited_image = cv2.addWeighted(original_image, 1, purple_mask, 1, 0)
    return composited_image

fig, axes = plt.subplots(2, 2)
img1_mask_add = mask_add(img_1_org, img_1_mask, (255, 250, 0))
img1_mask_add = cv2.cvtColor(img1_mask_add, cv2.COLOR_BGR2RGB)
img2_mask_add = mask_add(img_n, img_n_mask, (255, 250, 0))
img2_mask_add = cv2.cvtColor(img2_mask_add, cv2.COLOR_BGR2RGB)


axes[0][0].imshow(img_1_mask, cmap = 'gray')
axes[0][0].axis('off')
axes[0][0].set_title('first frame water mask')

axes[0][1].imshow(img_n_mask, cmap = 'gray')
axes[0][1].axis('off')
axes[0][1].set_title('second frame water mask')

axes[1][0].imshow(img1_mask_add)
axes[1][0].axis('off')
axes[1][0].set_title('first frame water mask')

axes[1][1].imshow(img2_mask_add)
axes[1][1].axis('off')
axes[1][1].set_title('second frame water mask')

plt.tight_layout()
plt.show()

## Flood Risk Estimation

### ROI-based Algorithm

#### Anchor-Point-based Function

In [ ]:
def find_white_point(img, x, y_start):
    height = img.shape[0]

    for y in range(y_start, height):
        if img[y, x] == 255:
            return (x, y)

    return None


total_water_px_dis = []
AP_results = []

x, y_start = std_cor


# 1. Single point
point1 = find_white_point(img_1_mask, x, y_start)
point2 = find_white_point(img_n_mask, x, y_start)

if point1 is None or point2 is None:
    print("Error: Cannot calculate water increment (single point)")
    total_water_px_dis.append(None)
    full_distance = None
else:
    total_water_px_dis.append(point1[1] - point2[1])
    full_distance = point1[1] - y_start


# 2. Three points
point_img1 = []
point_img2 = []

x_temp = x - 60

for i in range(3):
    if x_temp >= img_width:
        break

    p1 = find_white_point(img_1_mask, x_temp, y_start)
    p2 = find_white_point(img_n_mask, x_temp, y_start)

    if p1 is not None and p2 is not None:
        point_img1.append(p1[1])
        point_img2.append(p2[1])

    x_temp += 60

point_water_dis = []

for i in range(len(point_img1)):
    point_water_dis.append(point_img1[i] - point_img2[i])

if len(point_water_dis) > 0:
    total_water_px_dis.append(mean(point_water_dis))
else:
    print("Error: Cannot calculate water increment (three points)")
    total_water_px_dis.append(None)


# 3. Multiple anchor points
img_1_array = np.array(img_1_mask)
img_n_array = np.array(img_n_mask)

mask_height, mask_width = img_1_array.shape

height_list = []
x_array = []


for x_i in range(mask_width):
    for y in range(mask_height):
        if img_n_array[y, x_i] > 200 and img_1_array[y, x_i] > 200:
            x_array.append(x_i)
            break


for x_i in x_array:
    a = None
    b = None

    for y in range(mask_height):
        if img_1_array[y, x_i] > 200:
            a = y
            break

    for y in range(mask_height):
        if img_n_array[y, x_i] > 200:
            b = y
            break

    if a is not None and b is not None:
        height_list.append(a - b)

height = np.array(height_list)

valid_height = height[height > 0]

if len(valid_height) > 0:
    harmonic_height_mask = len(valid_height) / np.sum(1.0 / valid_height)
    total_water_px_dis.append(harmonic_height_mask)
else:
    print("No valid height data for harmonic mean calculation.")
    total_water_px_dis.append(None)


valid_results = []

for val in total_water_px_dis:
    if val is not None and val > 0:
        valid_results.append(val)


AP_results = []

if full_distance is not None and full_distance > 0:
    for val in valid_results:
        AP_results.append(val / full_distance * 100)
else:
    print("Error: invalid full distance")

print(f"Flood Conditions based on Anchor-Point ROI: {AP_results}")

#### Puddle-based Function

In [ ]:
Puddle_results_1 = []
Puddle_results_tmp = []
Puddle_results_2 = []
woong_num = int(len(ROI_cor)/4)

if woong_num == 0:
    print("Estimation based on Puddle-ROI is unavailable")
else:
    fig, axes = plt.subplots(woong_num, 2, figsize=(8, 8))
    if woong_num == 1:
        axes = [axes]
    for i in range(woong_num):
        roi = img_n_mask[ROI_cor[(i*4+0)][1]:ROI_cor[(i*4+2)][1], ROI_cor[(i*4+0)][0]:ROI_cor[(i*4+1)][0]]
        roi_ori = img_n_org[ROI_cor[(i*4+0)][1]:ROI_cor[(i*4+2)][1], ROI_cor[(i*4+0)][0]:ROI_cor[(i*4+1)][0]]
        percentage = (np.sum(roi)/255)/((ROI_cor[i*4][0]-ROI_cor[i*4+1][0])*(ROI_cor[i*4][1]-ROI_cor[i*4+2][1]))
        roi_ori = cv2.cvtColor(roi_ori, cv2.COLOR_BGR2RGB)
        axes[i][0].imshow(roi_ori)
        axes[i][0].axis('off')
        axes[i][0].set_title(f'Original Image2 ROI: {percentage:f}')
        axes[i][1].imshow(roi, cmap='gray', vmin=0, vmax=255)
        axes[i][1].axis('off')
        axes[i][1].set_title('Masked Image2 ROI')
        Puddle_results_1.append(percentage*100)
        

    plt.show()

    fig, axes = plt.subplots(woong_num, 2, figsize=(8, 8))
    if woong_num == 1:
        axes = [axes]
    for i in range(woong_num):
        roi = img_1_mask[ROI_cor[(i*4+0)][1]:ROI_cor[(i*4+2)][1], ROI_cor[(i*4+0)][0]:ROI_cor[(i*4+1)][0]]
        roi_ori = img_1_org[ROI_cor[(i*4+0)][1]:ROI_cor[(i*4+2)][1], ROI_cor[(i*4+0)][0]:ROI_cor[(i*4+1)][0]]
        percentage = (np.sum(roi)/255)/((ROI_cor[i*4][0]-ROI_cor[i*4+1][0])*(ROI_cor[i*4][1]-ROI_cor[i*4+2][1]))
        roi_ori = cv2.cvtColor(roi_ori, cv2.COLOR_BGR2RGB)
        axes[i][0].imshow(roi_ori)
        axes[i][0].axis('off')
        axes[i][0].set_title(f'Original Image1 ROI: {percentage:f}')
        axes[i][1].imshow(roi, cmap='gray', vmin=0, vmax=255)
        axes[i][1].axis('off')
        axes[i][1].set_title('Masked Image1 ROI')
        Puddle_results_tmp.append(percentage*100)

    plt.show()

    for i in range(len(Puddle_results_1)):
        Puddle_results_2.append(Puddle_results_1[i] - Puddle_results_tmp[i])

print("Flood Conditions based on Puddle ROI: ", Puddle_results_1)

#### CAR-based Function

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def preprocess_image(cropped_img):
    img_pil = Image.fromarray(cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB))
    img_tensor = preprocess(img_pil)
    img_tensor = img_tensor.unsqueeze(0).to(device)
    return img_tensor

def predict_cnn(model, cropped_car_images):
    predicted_classes = []
    predicted_probs = []
    with torch.no_grad():
        for img in cropped_car_images:
            img = preprocess_image(img)
            img = img.to(device)
            outputs = model(img)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted_class = torch.max(probabilities, 1)
            predicted_classes.append(predicted_class.item())
            predicted_probs.append(probabilities.cpu().numpy())
    return predicted_classes, predicted_probs

def prop_sorted_pre_class(A, B):
    prob_list = []
    sorted_class_list = []
    for i in range(len(A)):
        prob_list.append((B[i][0][A[i]], A[i]))
    prob_list.sort(reverse=True)
    for i in range(len(prob_list)):
        sorted_class_list.append(prob_list[i][1] + 1)
    return sorted_class_list


In [ ]:
Car_results_1 = []
Car_results_2 = []
Car_results_tmp = []

if len(cropped_car_images_2) == 0:
    print("there's no detected car")
else:
    path = './utils/model_weights/model_checkpoint_431_0.7600.pt'
    if os.path.exists(path):
        model = torch.load(path, map_location=device, weights_only=False)
        model = model.to(device)
        loss_function = nn.CrossEntropyLoss()
        model.eval()
    else:
        print("Checkpoint file can not be found.")
        exit()
    
    # Image2
    pred_class, pred_prob = predict_cnn(model, cropped_car_images_2)
    Car_results_1 = pred_class
    if len(Car_results_1) > 3:
        Car_results_1 = Car_results_1[-3:]    
    cropped_car_count = len(cropped_car_images_2)
    fig, axes = plt.subplots(1, cropped_car_count, figsize=((cropped_car_count*5), 5))
    if cropped_car_count == 1:
        axes = [axes]
    for i in range(cropped_car_count):
        cropped_car_images_2[i] = cv2.cvtColor(cropped_car_images_2[i], cv2.COLOR_BGR2RGB)
        if cropped_car_images_2[i] is not None and cropped_car_images_2[i].size > 0:
            axes[i].imshow(cropped_car_images_2[i])
            axes[i].axis('off')
            axes[i].set_title(f'Image {i+1} Predicted Level: {pred_class[i]+1}')
        else:
            print(f"Warning: Cropped Car Image {i+1} is None or empty")

    # Image1
    pred_class, pred_prob = predict_cnn(model, cropped_car_images_1)
    Car_results_tmp = pred_class
    if len(Car_results_tmp) > 3:
        Car_results_tmp = Car_results_tmp[-3:]
    cropped_car_count = len(cropped_car_images_1)
    fig, axes = plt.subplots(1, cropped_car_count, figsize=((cropped_car_count*5), 5))
    if cropped_car_count == 1:
        axes = [axes]
    for i in range(cropped_car_count):
        cropped_car_images_1[i] = cv2.cvtColor(cropped_car_images_1[i], cv2.COLOR_BGR2RGB)
        if cropped_car_images_1[i] is not None and cropped_car_images_1[i].size > 0:
            axes[i].imshow(cropped_car_images_1[i])
            axes[i].axis('off')
            axes[i].set_title(f'Image {i+1} Predicted Level: {pred_class[i]+1}')
        else:
            print(f"Warning: Cropped Car Image {i+1} is None or empty")

In [ ]:
for i in range(len(Car_results_1)):
    if Car_results_1[i] == 0:
        Car_results_1[i] = 0
    elif Car_results_1[i] == 1:
        Car_results_1[i] = 25
    elif Car_results_1[i] == 2:
        Car_results_1[i] = 50
    elif Car_results_1[i] == 3:
        Car_results_1[i] = 75
    else:
        Car_results_1[i] = 100

for i in range(len(Car_results_tmp)):
    if Car_results_tmp[i] == 0:
        Car_results_tmp[i] = 0
    elif Car_results_tmp[i] == 1:
        Car_results_tmp[i] = 25
    elif Car_results_tmp[i] == 2:
        Car_results_tmp[i] = 50
    elif Car_results_tmp[i] == 3:
        Car_results_tmp[i] = 75
    else:
        Car_results_tmp[i] = 100

if len(Car_results_1) != 0 and len(Car_results_tmp) != 0: 
    t2_avg = sum(Car_results_1) / len(Car_results_1)
    t1_avg = sum(Car_results_tmp) / len(Car_results_tmp)
    Car_results_2.append(t2_avg - t1_avg)
if len(Car_results_2) == 0:
    pass
else:
    if Car_results_2[0] < 0:
        Car_results_2 = []
if len(Car_results_1) <3:
    Car_results_1 =[]

print("Flood Conditions based on Car ROI: ", Car_results_1)

### Integration Algorithm

In [ ]:
ROI_weights = {
    'AP': 1.0,
    'Puddle': 1.0,
    'Car': 1.0
}

total_weight = sum(ROI_weights.values())

def normalized_weighted_sum(values, weight):
    numerator = 0
    denominator = 0
    for value in values:
        numerator = numerator + weight*value
        denominator = denominator + weight
    return numerator, denominator

normalized_AP = normalized_weighted_sum(AP_results, ROI_weights['AP'])
normalized_Puddle = normalized_weighted_sum(Puddle_results_1, ROI_weights['Puddle'])
normalized_Car = normalized_weighted_sum(Car_results_1, ROI_weights['Car'])
final_condition = (normalized_AP[0]+normalized_Puddle[0]+normalized_Car[0] )/ (normalized_AP[1]+normalized_Puddle[1]+normalized_Car[1])

normalized_AP2 = normalized_weighted_sum(AP_results, ROI_weights['AP'])
normalized_Puddle2 = normalized_weighted_sum(Puddle_results_2, ROI_weights['Puddle'])
normalized_Car2 = normalized_weighted_sum(Car_results_2, ROI_weights['Car'])
final_variation = (normalized_AP2[0]+normalized_Puddle2[0]+normalized_Car2[0] )/ (normalized_AP2[1]+normalized_Puddle2[1]+normalized_Car2[1])

In [ ]:
result_dict = {
    "temporal_difference": target_sec[1] - target_sec[0],
    "estimated_flood_condition": final_condition,
    "estimated_temporal_variation": final_variation,
    "anchor_point_roi": AP_results,
    "puddle_roi": {
        "value": Puddle_results_1,
        "variation": Puddle_results_2
    },
    "car_roi": {
        "value": Car_results_1,
        "variation": Car_results_2
    }
}

print(f"- Temporal difference of two given time points: {result_dict['temporal_difference']}\n"
      f"- Estimated Flood Condition:  {result_dict['estimated_flood_condition']} \n"
      f"- Estimated Temporal Variation:  {result_dict['estimated_temporal_variation']} \n"
      f"- Anchor-Point ROI: {result_dict['anchor_point_roi']}\n"
      f"- Puddle ROI: {result_dict['puddle_roi']['value']} (variation: {result_dict['puddle_roi']['variation']})\n"
      f"- Car ROI: {result_dict['car_roi']['value']} (variation: {result_dict['car_roi']['variation']})"
     )

with open("./outputs/result.json", "w", encoding="utf-8") as f:
    json.dump(result_dict, f, ensure_ascii=False, indent=4)

img_1_copy2_show = cv2.cvtColor(img_1_copy2, cv2.COLOR_BGR2RGB)
plt.imshow(img_1_copy2_show)
plt.axis('off')
plt.show()

In [ ]:
def save_with_title(image, title, filename, gray=False):
    fig, ax = plt.subplots(figsize=(8, 8))
    if gray:
        ax.imshow(image, cmap ='gray')
    else:
        ax.imshow(image)
    ax.set_title(title, fontsize=16)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()
output_directory = './outputs/images'
os.makedirs(output_directory, exist_ok=True)

img_1_add_mask = cv2.cvtColor(mask_add(img_1_org, img_1_mask, (0, 100, 55)), cv2.COLOR_BGR2RGB)
img_n_add_mask = cv2.cvtColor(mask_add(img_n_org, img_n_mask, (0, 100, 55)), cv2.COLOR_BGR2RGB)
img_1_copy2_show = cv2.cvtColor(img_1_copy2, cv2.COLOR_BGR2RGB)

save_with_title(img_1_mask, 'Water Mask of First Frame Image', os.path.join(output_directory, 'img_1_mask.png'), gray=True)
save_with_title(img_n_mask, 'Water Mask of Second Frame Image',  os.path.join(output_directory, 'img_2_mask.png'), gray=True)
save_with_title(img_1_add_mask, 'First Frame Image with Water Mask', os.path.join(output_directory, 'img_1_add_mask.png'))
save_with_title(img_n_add_mask, 'Second Frame Image with Water Mask',  os.path.join(output_directory, 'img_2_add_mask.png'))
save_with_title(img_1_copy2_show, 'Selected ROIs',  os.path.join(output_directory, 'img_1_copy2_show.png'))

if car_detected_img1 is not None and car_detected_img1.size:
    save_with_title(detected_img1, 'Detected Cars',  os.path.join(output_directory, 'car_detected_img1.png'))
if car_detected_img2 is not None and car_detected_img2.size:
    save_with_title(detected_img2, 'Detected Cars',  os.path.join(output_directory, 'car_detected_img2.png'))